# Experiment 4: Raw FAISS Vector Retrieval Baseline (No LangChain)

**Curriculum Focus**: Laboratory Experiment 4 Retrieval Baseline.
**Objective**: Implement Dense Vector Retrieval strictly from first principles using:
- Paragraph-based chunking
- `SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')`
- Native `faiss.IndexFlatL2`
- Manual prompt formatting
- Generative grounding with `Qwen/Qwen2.5-0.5B-Instruct`

### Purpose:
Proves the RAG vector space principles from scratch before introducing LangChain abstractions in `app/`.

In [1]:
# Step 1: Install Dependencies
!pip install -q sentence-transformers faiss-cpu transformers torch tabulate


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: C:\Users\mahir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np
import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

ModuleNotFoundError: No module named 'faiss'

## 1. Load Raw Documents and Chunk by Paragraphs
We read raw text files and split on double newlines `\n\n`.

In [ ]:
documents = [
    {'source': 'lab_manual_fine_tuning.pdf', 'text': 'LoRA decomposes weight updates Delta W into B @ A with rank r and alpha scaling factor. Pretrained weights W_0 are frozen.'},
    {'source': 'lab_manual_fine_tuning.pdf', 'text': 'QLoRA introduces 4-bit NormalFloat (NF4), double quantization to save 0.37 bits per parameter, and paged optimizers.'},
    {'source': 'lecture_slides_rag_and_embeddings.pdf', 'text': 'Dense bi-encoders encode queries and documents independently. all-MiniLM-L6-v2 produces 384-dimensional dense embeddings.'},
    {'source': 'lecture_slides_rag_and_embeddings.pdf', 'text': 'faiss.IndexFlatL2 computes exact Euclidean distance with 100% recall. Chunk size 500 with overlap 80 prevents boundary loss.'},
    {'source': 'course_notes_orchestration_and_apis.txt', 'text': 'LangGraph models stateful workflows with nodes and conditional edges. Bounded retry checks retry_count < 1 to prevent runaway loops.'}
]

corpus_texts = [d['text'] for d in documents]
print(f'Prepared {len(corpus_texts)} paragraph chunks.')

## 2. Compute Embeddings and Build Native FAISS IndexFlatL2

In [ ]:
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = embedder.encode(corpus_texts, convert_to_numpy=True, normalize_embeddings=True)
dimension = embeddings.shape[1]
print(f'Embedding Dimension: {dimension}')

# Initialize exact L2 flat FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f'Total vectors added to FAISS index: {index.ntotal}')

## 3. Manual Retrieval Function (k = 1, 2, 3)

In [ ]:
def raw_search(query, k=3):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = index.search(q_emb, k)
    retrieved_chunks = []
    for i, idx in enumerate(indices[0]):
        retrieved_chunks.append({
            'rank': i + 1,
            'chunk_id': int(idx),
            'source': documents[idx]['source'],
            'text': documents[idx]['text'],
            'distance': float(distances[0][i])
        })
    return retrieved_chunks

## 4. Experiment 4 Observation Study
We query for k=1, 2, 3 and record how retrieved chunks and contextual support change.

In [ ]:
test_queries = [
    'What is the formula for LoRA weight updates?',
    'How does FAISS compute vector similarity?',
    'What is the capital of France?' # out-of-scope test
]

obs_records = []
for q in test_queries:
    for k_val in [1, 2, 3]:
        hits = raw_search(q, k=k_val)
        top_sources = ', '.join([f'{h["source"]}(d={h["distance"]:.2f})' for h in hits])
        supported = 'Yes' if any(w in hits[0]['text'].lower() for w in ['lora', 'faiss']) and 'capital' not in q else 'No'
        obs_records.append({
            'Query': q,
            'k': k_val,
            'Top Chunks Retrieved': len(hits),
            'Retrieved Sources': top_sources,
            'Supported by Context?': supported
        })

df_obs = pd.DataFrame(obs_records)
print(df_obs.to_markdown(index=False))
os.makedirs('eval/results', exist_ok=True)
df_obs.to_csv('eval/results/exp4_faiss_baseline_study.csv', index=False)
print('Saved Experiment 4 observation table to eval/results/exp4_faiss_baseline_study.csv')